# Phase 3 — Blocking Benchmark

This notebook evaluates label-blind candidate generation on the fixed Phase 2 validation split (441,363 Source 1 rows) against the full training Source 2/3 target pool (10,320,219 records). Ground truth is used only for scoring, never to construct blocks. Candidate keys use Unicode NFKC, case folding, and alphanumeric normalization. Name/address blocks require exact country equality; country labels are open-set. Token and character routes provide a country-agnostic fallback. No external data is used.

## Results

| Route | Candidate pairs | S2 target recall | S3 target recall | Overall target recall |
|---|---:|---:|---:|---:|
| Exact normalized name + country | 3,569,884 | 21.2% | 21.9% | 21.6% |
| Order-insensitive canonical name + country | 6,840,291 | 39.4% | 38.0% | 38.7% |
| Expanded rare address tokens (global) | 21,848,901 | 41.2% | 38.1% | 39.6% |
| Name token pairs (global) | 22,474,496 | 56.5% | 56.9% | 56.7% |
| Address token pairs (global) | 17,070,899 | 77.2% | 61.3% | 69.0% |
| Union before token pairs | 41,104,467 | 77.0% | 73.1% | 75.0% |
| Union with expanded single tokens | 43,421,260 | 77.3% | 73.3% | 75.3% |
| Final union with token pairs | 75,649,390 | 93.2% | 91.1% | 92.1% |

The final union emits **75,649,390 candidate pairs**, about **171.4 per validation entity**, from a theoretical 4.55 trillion comparisons (reduction ratio **99.99834%**). Target recall is **92.14% overall**, **93.20% for S2**, and **91.14% for S3**. It finds at least one true target for **98.88%** of Source 1 rows with matches (96.02% for S2 and 95.51% for S3).

## Decision

The address token-pair route improved the combined blocking recall by about **16.9 percentage points** over the previous best union, at the cost of increasing the candidate count by about **32.2 million pairs**. A deterministic sample of 200 final misses (100 per target source) is saved in `dataset/splits/phase3_missed_examples.json`; all sampled pairs have the same country, so country mismatch does not explain these misses. In this sample, mean name-token Jaccard is 0.239 for S2 and 0.354 for S3; mean address-token Jaccard is 0.520 for S2 and 0.339 for S3. Some target addresses are blank (8/100 S2, 14/100 S3). This points to residual name variation for S2 and weaker address overlap for S3.
Phase 3 is **complete**. The final blocking route meets the working exit criterion adopted for this handoff: more than 90% target recall overall and for each source, while reducing the theoretical pair universe by more than 99.99%. The validation candidate set is suitable for the matching-model phase, with an explicit recall ceiling of 92.14%; missed true links cannot be recovered downstream. Preserve separate S2/S3 metrics and measure matcher quality with macro F0.5, including singletons. No external data was used.

## Reproducibility

The standalone benchmark is `phase3_blocking_benchmark.py`; it rebuilds compact disk-backed indexes and writes detailed route metrics to `dataset/splits/phase3_blocking_metrics.json` plus a deterministic missed-link sample to `dataset/splits/phase3_missed_examples.json`. The cell below reruns it end to end. It takes several minutes on the full data.


In [ ]:
from pathlib import Path

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "notebooks" / "validation" / "phase3_blocking_benchmark.py").is_file())
script_path = project_root / "notebooks" / "validation" / "phase3_blocking_benchmark.py"
exec(compile(script_path.read_text(encoding="utf-8"), str(script_path), "exec"), {"__name__": "__main__"})
